# Prakiraan CO satu jam ke depan
**Kecerdasan Buatan TE2514031 · Kelas B · Kelompok 9**

Notebook mandiri untuk Google Colab atau Jupyter. Jalankan sel berurutan dari atas. Sumber data [UCI Air Quality](https://doi.org/10.24432/C59K5F) diunduh saat eksperimen. Kode pelatihan, checkpoint, dan inferensi memakai implementasi yang sama dengan dashboard.

## 1. Siapkan lingkungan
Python 3.11 atau 3.12 dianjurkan. Sel ini membuat folder kerja untuk satu sesi. Paket dasar dipasang jika belum tersedia dalam versi eksperimen.

In [ ]:
from pathlib import Path
from importlib import metadata
from tempfile import mkdtemp
import os, sys, subprocess

needed = {"numpy": "2.3.5", "pandas": "2.2.3"}
def local_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None
missing = [f"{k}=={v}" for k, v in needed.items() if local_version(k) != v]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
work = Path(mkdtemp(prefix="kelompok9-co-"))
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
print("Lingkungan eksperimen siap.")

## 2. Implementasi yang digunakan kelompok
Enam modul di bawah menyimpan pembaca data, RNN dengan backpropagation through time, GA/PSO, inferensi fuzzy, pembaca checkpoint, dan prosedur eksperimen di folder sesi. Parameter pencarian dan pembentukan jendela dapat diperiksa langsung.

### air_data.py

In [ ]:
(work / 'air_data.py').write_text(r'''"""Hourly UCI observations, chronological partitions and train-only transforms."""
from hashlib import sha256
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd

UCI_URL = "https://archive.ics.uci.edu/static/public/360/air+quality.zip"
UCI_CSV_SHA256 = "13277ae5d8581e80b7be09d47c7d3d06fe9b8e957078f2cf6e859f955e62f996"
TARGET = "CO(GT)"
FEATURES = ["PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)",
            "PT08.S5(O3)", "T", "RH", "AH"]
MIN_STEPS, MAX_STEPS = 4, 8


def official_csv(cache):
    path = Path(cache)
    if path.exists():
        csv = path.read_bytes()
    else:
        with urlopen(UCI_URL, timeout=60) as response, ZipFile(BytesIO(response.read())) as archive:
            csv = archive.read("AirQualityUCI.csv")
        if sha256(csv).hexdigest() == UCI_CSV_SHA256:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(csv)
    if sha256(csv).hexdigest() != UCI_CSV_SHA256:
        raise ValueError("Berkas cache berbeda dari CSV UCI untuk eksperimen ini.")
    return csv


def numbers(series):
    text = series.astype("string").str.strip().replace("", pd.NA).str.replace(",", ".", regex=False)
    values = pd.to_numeric(text, errors="coerce").astype(float)
    if (text.notna() & values.isna()).any() or np.isinf(values).any():
        raise ValueError(f"Kolom {series.name} berisi angka yang tidak valid.")
    return values.replace(-200, np.nan)


def read_data(file, required, require_contiguous=True):
    """Accept the original UCI CSV or an ISO timestamp CSV, including forecast-only inputs."""
    data = file if isinstance(file, pd.DataFrame) else pd.read_csv(
        file, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
    data = data.dropna(how="all").copy()
    data.columns = data.columns.str.strip()
    data = data.loc[:, ~data.columns.str.startswith("Unnamed:")]
    if data.empty:
        raise ValueError("CSV tidak memuat baris observasi.")
    if data.columns.duplicated().any():
        raise ValueError("Nama kolom harus unik.")
    if {"Date", "Time"} <= set(data.columns):
        dates = data.pop("Date").astype(str) + " " + data.pop("Time").astype(str)
        times = pd.to_datetime(dates, format="%d/%m/%Y %H.%M.%S", errors="coerce")
    else:
        key = next((k for k in ["timestamp", "Date_Time"] if k in data), None)
        if key is None:
            raise ValueError("Sertakan timestamp ISO, atau pasangan Date dan Time seperti pada CSV UCI.")
        times = pd.to_datetime(data.pop(key), format="ISO8601", errors="coerce")
    if times.isna().any() or times.duplicated().any():
        raise ValueError("Ada waktu kosong, tidak valid, atau duplikat.")
    if not times.dt.floor("h").equals(times):
        raise ValueError("Setiap waktu observasi harus tepat pada jam penuh.")
    absent = [name for name in required if name not in data]
    if absent:
        raise ValueError("Kolom wajib belum ada: " + ", ".join(absent))
    values = data.apply(numbers)
    values.index = pd.DatetimeIndex(times, name="timestamp")
    values = values.sort_index()
    if require_contiguous and not values.index.to_series().diff().dropna().eq(pd.Timedelta(hours=1)).all():
        raise ValueError("Riwayat harus berurutan per jam. Isi jam yang hilang sebagai baris kosong beserta timestamp.")
    return values


def fit_transform(frame, end_train):
    training = frame.iloc[:end_train]
    x = training[FEATURES]
    med = x.median()
    if med.isna().any() or training[TARGET].dropna().empty:
        raise ValueError("Data latih tidak cukup untuk menghitung parameter.")
    filled = x.fillna(med)
    y = training[TARGET].dropna()
    return {"features": FEATURES, "median": med.tolist(), "mean": filled.mean().tolist(),
            "std": filled.std(ddof=0).replace(0, 1).tolist(), "y_mean": float(y.mean()),
            "y_std": float(y.std(ddof=0)) or 1.0}


def standardize(frame, columns, prep):
    ids = [prep["features"].index(c) for c in columns]
    raw = frame[columns].to_numpy(dtype=float)
    filled = np.where(np.isnan(raw), np.asarray(prep["median"])[ids], raw)
    return ((filled - np.asarray(prep["mean"])[ids]) / np.asarray(prep["std"])[ids]).astype("float32")


def make_windows(arr, length):
    return np.lib.stride_tricks.sliding_window_view(arr, length, axis=0).swapaxes(1, 2)


def prepare(frame):
    n = len(frame)
    a, b = int(n * .70), int(n * .85)
    prep = fit_transform(frame, a)
    x = standardize(frame, FEATURES, prep)
    target = frame[TARGET].to_numpy()
    # A target at i only sees inputs through i-1. Every model uses the same target times.
    anchors = np.arange(MAX_STEPS, n)
    present = make_windows(frame[FEATURES].notna().to_numpy(), MIN_STEPS)
    usable = present[anchors - MIN_STEPS].any(axis=1).all(axis=1)
    anchors = anchors[usable & np.isfinite(target[anchors])]
    splits = {"train": anchors[anchors < a], "val": anchors[(anchors >= a) & (anchors < b)],
              "test": anchors[anchors >= b]}
    if any(len(part) < 2 for part in splits.values()):
        raise ValueError("Setiap partisi memerlukan sedikitnya dua target CO dengan riwayat sensor.")
    return prep, x, target, splits, (a, b)
''', encoding='utf-8')
print('air_data.py siap.')

### recurrent.py

In [ ]:
(work / 'recurrent.py').write_text(r'''"""Elman RNN with backpropagation through time, Adam, and portable NumPy weights."""
import numpy as np

BASE = {"length": 6, "units": 12, "lr": .003, "dropout": .10}


def init_params(n_features, hidden, rng):
    def weight(a, b):
        lim = np.sqrt(6 / (a + b))
        return rng.uniform(-lim, lim, (a, b)).astype("float32")
    return {"wx": weight(n_features, hidden), "wh": weight(hidden, hidden) * .65,
            "bh": np.zeros(hidden, "float32"), "wy": weight(hidden, 1).ravel(),
            "by": np.zeros(1, "float32")}


def hidden_states(x, p):
    states = [np.zeros((len(x), len(p["bh"])), dtype="float32")]
    for step in x.transpose(1, 0, 2):
        states.append(np.tanh(step @ p["wx"] + states[-1] @ p["wh"] + p["bh"]))
    return states


def predict(x, p):
    return hidden_states(x, p)[-1] @ p["wy"] + p["by"]


def fit(x, y, vx, vy, config, seed, epochs, patience):
    rng = np.random.default_rng(seed)
    p = init_params(x.shape[-1], config["units"], rng)
    m, v = ({key: np.zeros_like(val) for key, val in p.items()} for _ in range(2))
    history, best, best_loss = [], {}, np.inf
    step = stale = 0
    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(x))
        for start in range(0, len(order), 128):
            ids = order[start:start + 128]
            batch = x[ids]
            states = hidden_states(batch, p)
            keep = 1 - config["dropout"]
            mask = (rng.random(states[-1].shape) < keep).astype("float32") / keep
            output = states[-1] * mask
            error = (2 / len(ids)) * (output @ p["wy"] + p["by"] - y[ids])
            grads = {"wy": output.T @ error, "by": np.atleast_1d(error.sum()),
                     "wx": np.zeros_like(p["wx"]), "wh": np.zeros_like(p["wh"]),
                     "bh": np.zeros_like(p["bh"])}
            dh = error[:, None] * p["wy"] * mask
            for t in reversed(range(batch.shape[1])):
                dz = dh * (1 - states[t + 1] ** 2)
                grads["wx"] += batch[:, t].T @ dz
                grads["wh"] += states[t].T @ dz
                grads["bh"] += dz.sum(axis=0)
                dh = dz @ p["wh"].T
            step += 1
            for key, grad in grads.items():
                grad = np.clip(grad, -5., 5.)
                m[key] = .9 * m[key] + .1 * grad
                v[key] = .999 * v[key] + .001 * grad ** 2
                p[key] -= config["lr"] * (m[key] / (1 - .9 ** step)) / (np.sqrt(v[key] / (1 - .999 ** step)) + 1e-8)
        loss = float(np.mean((predict(x, p) - y) ** 2))
        val_loss = float(np.mean((predict(vx, p) - vy) ** 2))
        history.append({"epoch": epoch, "train_mse_z": loss, "val_mse_z": val_loss})
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            best = {k: a.copy() for k, a in p.items()}
            best_epoch = epoch
        else:
            stale += 1
            if stale >= patience:
                break
    return best, history, best_epoch
''', encoding='utf-8')
print('recurrent.py siap.')

### optimize.py

In [ ]:
(work / 'optimize.py').write_text(r'''"""Validation-only evolutionary searches. One search per seed with cumulative logs."""
import numpy as np

from air_data import FEATURES
from recurrent import BASE

POP, GENERATIONS = 6, 5
PARTICLES, ROUNDS = 5, 5
SEEDS = (7, 19, 42)
PENALTY = .03
LENGTHS, UNITS = (4, 6, 8), (8, 12, 16, 20, 24)


def genetic(score, seed):
    rng = np.random.default_rng(seed)
    n = len(FEATURES)
    pop = rng.integers(0, 2, (POP, n)).astype(bool)
    pop[0] = True
    history, records = [], []
    for gen in range(1, GENERATIONS + 1):
        for mask in pop:
            if not mask.any():
                mask[rng.integers(n)] = True
        fitness = []
        for mask in pop:
            error = score(tuple(np.flatnonzero(mask)), BASE)
            cost = error + PENALTY * mask.sum() / n
            fitness.append(cost)
            records.append({"generation": gen, "bits": "".join(map(str, mask.astype(int))),
                            "val_rmse": error, "fitness": float(cost)})
        elite = pop[np.argmin(fitness)].copy()
        history.append(float(min(fitness)))
        new = [elite]
        def tournament():
            ids = rng.choice(POP, size=3, replace=False)
            return pop[ids[np.argmin(np.asarray(fitness)[ids])]]
        while len(new) < POP:
            left, right = tournament(), tournament()
            child = left.copy()
            if rng.random() < .8:
                cut = rng.integers(1, n)
                child[cut:] = right[cut:]
            child ^= rng.random(n) < 1 / n
            new.append(child)
        pop = np.asarray(new)
    return {"seed": seed, "features": [FEATURES[i] for i in np.flatnonzero(elite)],
            "fitness": history[-1], "history": history, "candidates": records}


def decode(position):
    length, units, rate, drop = position
    return {"length": LENGTHS[round(float(length) * (len(LENGTHS) - 1))],
            "units": UNITS[round(float(units) * (len(UNITS) - 1))],
            "lr": round(float(10 ** (-3 + rate)), 5), "dropout": round(.25 * float(drop), 2)}


def swarm(score, indices, seed):
    rng = np.random.default_rng(seed)
    pos = rng.random((PARTICLES, 4))
    vel = rng.uniform(-.15, .15, size=pos.shape)
    pbest, p_cost = pos.copy(), np.full(PARTICLES, np.inf)
    history, records = [], []
    for iteration in range(1, ROUNDS + 1):
        for i, particle in enumerate(pos):
            params = decode(particle)
            cost = score(indices, params)
            records.append({"iteration": iteration, "particle": i + 1, "params": params,
                            "position": particle.tolist(), "val_rmse": cost})
            if cost < p_cost[i]:
                p_cost[i], pbest[i] = cost, particle.copy()
        best = pbest[np.argmin(p_cost)].copy()
        history.append(float(np.min(p_cost)))
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = np.clip(.65 * vel + 1.4 * r1 * (pbest - pos) + 1.4 * r2 * (best - pos), -.35, .35)
        pos = np.clip(pos + vel, 0, 1)
    return {"seed": seed, "parameters": decode(best), "fitness": history[-1],
            "history": history, "candidates": records}
''', encoding='utf-8')
print('optimize.py siap.')

### fuzzy_system.py

In [ ]:
(work / 'fuzzy_system.py').write_text(r'''"""Single-input Mamdani FIS for a project-specific CO monitoring priority."""
import numpy as np

LEVELS = ["Rendah", "Terpantau", "Meningkat", "Tinggi", "Sangat tinggi"]
ACTIONS = ["Catat hasil pemantauan rutin.", "Amati tren pada jam berikutnya.",
           "Bandingkan hasil dengan pengukuran referensi.", "Prioritaskan verifikasi pengukuran.",
           "Segera periksa sensor dan ulangi pengukuran referensi."]
IN_CENTERS = [0., 2., 4., 7., 10.]
OUT_CENTERS = [0., 25., 50., 75., 100.]
GRID = np.linspace(0., 100., 201)


def memberships(values, centers):
    v = np.asarray(values, dtype=float)
    if not np.isfinite(v).all():
        raise ValueError("Masukan fuzzy harus berupa angka hingga.")
    c = np.array(centers)
    # Linear partitions with shoulders at the two ends.
    return np.stack([np.interp(v, c, row) for row in np.eye(len(c))], axis=-1)


def fuzzy_detail(co):
    if not np.isfinite(co) or co < 0:
        raise ValueError("Konsentrasi CO harus berupa angka nonnegatif.")
    degrees = memberships(float(co), IN_CENTERS)
    consequent = memberships(GRID, OUT_CENTERS)
    aggregate = np.minimum(degrees, consequent).max(axis=1)
    score = float(np.trapezoid(GRID * aggregate, GRID) / np.trapezoid(aggregate, GRID))
    idx = int(np.argmin(abs(np.array(OUT_CENTERS) - score)))
    return score, idx, degrees, aggregate


def compute_fuzzy_risk(co_value):
    score, idx, _, _ = fuzzy_detail(float(co_value))
    return score, LEVELS[idx], ["🟢", "🟡", "🟠", "🔴", "🟣"][idx], ACTIONS[idx]
''', encoding='utf-8')
print('fuzzy_system.py siap.')

### inference.py

In [ ]:
(work / 'inference.py').write_text(r'''"""Shared interface for the dashboard, notebook and command-line checks."""
import json
from pathlib import Path
import numpy as np
import pandas as pd

from air_data import MIN_STEPS, make_windows, read_data, standardize
from fuzzy_system import compute_fuzzy_risk
from recurrent import predict

DEFAULT = Path(__file__).resolve().parent / "artifacts" / "uci-v1"


def load_run(directory=DEFAULT):
    directory = Path(directory)
    return json.loads((directory / "experiment.json").read_text(encoding="utf-8"))


def estimate(x, weights, prep):
    return np.maximum(0, predict(x, weights) * prep["y_std"] + prep["y_mean"])


def forecast(source, directory=DEFAULT):
    directory = Path(directory)
    result = load_run(directory)
    spec = result["models"]["hybrid"]
    cols, length = spec["features"], spec["params"]["length"]
    frame = read_data(source, cols)
    if len(frame) < length:
        raise ValueError(f"Sediakan setidaknya {length} jam observasi.")
    inputs = make_windows(standardize(frame, cols, result["preprocessing"]), length)
    # Require per-channel evidence in the last four hours of each input.
    recent = make_windows(frame[cols].notna().to_numpy(), MIN_STEPS)[length - MIN_STEPS:]
    valid = recent.any(axis=1).all(axis=1)
    dates = frame.index[length - 1:] + pd.Timedelta(hours=1)
    values = np.full(len(dates), np.nan)
    if valid.any():
        with np.load(directory / "hybrid.npz", allow_pickle=False) as weights:
            values[valid] = estimate(inputs[valid], weights, result["preprocessing"])
    rows = []
    for date, co in zip(dates, values):
        if np.isfinite(co):
            score, level, _, action = compute_fuzzy_risk(co)
            rows.append((date, co, score, level, action))
        else:
            rows.append((date, np.nan, np.nan, "Data kurang", "Lengkapi riwayat sensor empat jam terakhir."))
    return pd.DataFrame(rows, columns=["timestamp_target", "CO_mg_m3", "skor", "prioritas", "tindak_lanjut"])
''', encoding='utf-8')
print('inference.py siap.')

### train.py

In [ ]:
(work / 'train.py').write_text(r'''"""Train and seal a self-contained evidence bundle for a one-hour CO forecast."""
import argparse
from datetime import datetime
from functools import lru_cache
from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import numpy as np
import pandas as pd

from air_data import FEATURES, MIN_STEPS, MAX_STEPS, TARGET, make_windows, official_csv, prepare, read_data
from inference import estimate
from optimize import PENALTY, SEEDS, genetic, swarm
from recurrent import BASE, fit

MODELS = {"baseline": "RNN dasar", "ga": "GA + RNN", "pso": "PSO + RNN", "hybrid": "GA + PSO + RNN"}
SEARCH_N, SEARCH_EPOCHS, FINAL_EPOCHS, PATIENCE = 1024, 5, 32, 7


def metrics(y, p):
    residual = p.astype(float) - y
    total = np.sum((y - y.mean()) ** 2)
    return {"mae": float(np.abs(residual).mean()), "rmse": float(np.sqrt(np.mean(residual ** 2))),
            "r2": float(1 - (residual @ residual) / total) if total > 0 else None,
            "bias": float(residual.mean())}


def data_summary(frame, anchors, cuts, folder):
    result = []
    for col, series in frame.items():
        values = series.dropna()
        hist, edges = np.histogram(values, bins=18) if len(values) else ([], [])
        result.append({"variabel": col, "missing": int(series.isna().sum()), "observed": int(len(values)),
                       "min": float(values.min()) if len(values) else None,
                       "median": float(values.median()) if len(values) else None,
                       "max": float(values.max()) if len(values) else None,
                       "hist": np.asarray(hist).tolist(), "edges": np.asarray(edges).tolist()})
    frame.resample("D").mean().to_csv(folder / "daily.csv", float_format="%.5f")
    splits = {}
    boundaries = (0, *cuts, len(frame))
    for n, (key, ids) in enumerate(anchors.items()):
        left, right = boundaries[n:n+2]
        splits[key] = {"from": str(frame.index[left]), "to": str(frame.index[right-1]), "hours": right-left,
                       "observed_target": int(frame[TARGET].iloc[left:right].count()), "n": len(ids)}
    return {"rows": len(frame), "raw_columns": len(frame.columns) + 2,
            "summary": result, "splits": splits, "from": str(frame.index[0]), "to": str(frame.index[-1])}


def run(frame, out, digest, seeds=SEEDS):
    out = Path(out).resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        raise FileExistsError(f"Gunakan direktori keluaran baru. {out.name} sudah ada.")
    prep, x, y, parts, cuts = prepare(frame)
    yn = ((y - prep["y_mean"]) / prep["y_std"]).astype("float32")
    tr, va, te = (parts[k] for k in ["train", "val", "test"])
    all_cols = tuple(range(len(FEATURES)))
    pilot = tr[np.linspace(0, len(tr)-1, min(len(tr), SEARCH_N), dtype=int)]
    windows = {l: make_windows(x, l) for l in range(MIN_STEPS, MAX_STEPS+1)}

    def take(ids, ix, length):
        return np.ascontiguousarray(windows[length][ids-length][:, :, list(ix)])

    @lru_cache(maxsize=None)
    def cached(ix, config):
        cfg = dict(config)
        xt, xv = (take(k, ix, cfg["length"]) for k in (pilot, va))
        weights, _, _ = fit(xt, yn[pilot], xv, yn[va], cfg, 271, SEARCH_EPOCHS, SEARCH_EPOCHS)
        return metrics(y[va], estimate(xv, weights, prep))["rmse"]

    def objective(ix, params):
        return cached(tuple(ix), tuple(sorted(params.items())))

    def timed_search(name, procedure):
        records = []
        for seed in seeds:
            start = perf_counter()
            hits = cached.cache_info().misses
            record = procedure(seed)
            record["seconds"] = perf_counter() - start
            record["new_fits"] = cached.cache_info().misses - hits
            records.append(record)
            print(name, seed, round(record["fitness"], 5), flush=True)
        return records

    ga = timed_search("GA", lambda s: genetic(objective, s))
    features = min(ga, key=lambda r: r["fitness"])["features"]
    selected = tuple(FEATURES.index(c) for c in features)
    pso_full = timed_search("PSO 8", lambda s: swarm(objective, all_cols, s))
    pso_ga = timed_search("PSO GA", lambda s: swarm(objective, selected, s))
    get_params = lambda log: min(log, key=lambda r: r["fitness"])["parameters"]
    specs = {"baseline": (all_cols, BASE), "ga": (selected, BASE),
             "pso": (all_cols, get_params(pso_full)), "hybrid": (selected, get_params(pso_ga))}
    with TemporaryDirectory(dir=out.parent) as temp:
        folder = Path(temp) / "bundle"
        folder.mkdir()
        result = {"source_sha256": digest, "preprocessing": prep, "data": data_summary(frame, parts, cuts, folder),
                  "seeds": list(seeds), "search_train": len(pilot), "search_epochs": SEARCH_EPOCHS,
                  "search_init_seed": 271, "final_init_seed": 42, "max_epochs": FINAL_EPOCHS,
                  "patience": PATIENCE, "ga_penalty": PENALTY, "ga": ga, "pso_full": pso_full, "pso_ga": pso_ga,
                  "models": {}}
        # Freeze every model and checkpoint with train/validation before scoring any test predictions.
        for key, (ix, cfg) in specs.items():
            start = perf_counter()
            xt, xv = (take(ids, ix, cfg["length"]) for ids in (tr, va))
            weight, history, best_epoch = fit(xt, yn[tr], xv, yn[va], cfg, 42, FINAL_EPOCHS, PATIENCE)
            np.savez_compressed(folder / (key + ".npz"), **weight)
            result["models"][key] = {"label": MODELS[key], "features": [FEATURES[i] for i in ix],
                "params": cfg, "val": metrics(y[va], estimate(xv, weight, prep)), "best_epoch": best_epoch,
                "history": history, "fit_seconds": perf_counter()-start}
        predictions = pd.DataFrame({"timestamp_target": frame.index[te], "aktual": y[te]})
        for key, (ix, cfg) in specs.items():
            with np.load(folder / (key + ".npz"), allow_pickle=False) as checkpoint:
                preds = estimate(take(te, ix, cfg["length"]), checkpoint, prep)
            predictions[key] = preds
            result["models"][key]["test"] = metrics(y[te], preds)
        previous = frame[TARGET].shift(1).ffill().fillna(frame[TARGET].iloc[:cuts[0]].median())
        predictions["persistensi"] = previous.iloc[te].to_numpy()
        result["persistence"] = metrics(y[te], predictions["persistensi"].to_numpy())
        predictions.to_csv(folder / "test_predictions.csv", index=False)
        length = specs["hybrid"][1]["length"]
        demo_id = next((i for i in te if i >= 24 and frame[features].iloc[i-24:i].notna().all().all()), None)
        if demo_id is None:
            demo_id = int(te[0])
        demo = frame[features].iloc[demo_id-length:demo_id]
        demo.to_csv(folder / "contoh_sensor.csv")
        result["demo_target"] = str(frame.index[demo_id])
        (folder / "experiment.json").write_text(json.dumps(result, allow_nan=False, ensure_ascii=False, indent=2),
                                               encoding="utf-8")
        folder.rename(out)
    return out


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--csv", type=Path, help="CSV UCI atau file data latih yang kompatibel")
    parser.add_argument("--out", type=Path, default=None)
    args = parser.parse_args()
    raw = args.csv.read_bytes() if args.csv else official_csv(".cache/AirQualityUCI.csv")
    frame = read_data(BytesIO(raw), [*FEATURES, TARGET])
    destination = args.out or Path("runs") / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    print(run(frame, destination, sha256(raw).hexdigest()))


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('train.py siap.')

## 3. Latih dan evaluasi empat konfigurasi
Jalankan proses Python baru agar versi dependensi dan pengaturan CPU berlaku penuh. Pencarian hanya memakai data latih dan validasi. Set uji dibuka setelah keempat checkpoint dibekukan.

In [ ]:
from datetime import datetime
run_dir = work / ("run-" + datetime.now().strftime("%H%M%S-%f"))
subprocess.run([sys.executable, "-u", "train.py", "--out", str(run_dir)], cwd=work, check=True)
print("Empat checkpoint sudah disimpan.")

## 4. Baca tabel hasil dan kurva kesalahan
MAE dan RMSE dinyatakan dalam mg/m³. Semua baris pada tabel perbandingan memakai waktu uji yang sama. Persistensi memakai CO teramati terakhir.

In [ ]:
import json
import pandas as pd
from IPython.display import display

exp = json.loads((run_dir / "experiment.json").read_text())
display(pd.DataFrame(exp["data"]["splits"]).T)
scores = pd.DataFrame({v["label"]: v["test"] for v in exp["models"].values()}).T
scores.loc["Persistensi"] = exp["persistence"]
display(scores.round(4))
print("Kanal terpilih:", ", ".join(exp["models"]["hybrid"]["features"]))

In [ ]:
import matplotlib.pyplot as plt

pr = pd.read_csv(run_dir / "test_predictions.csv", index_col="timestamp_target", parse_dates=True)
first_week = pr.loc[:pr.index.min() + pd.Timedelta(hours=167)].asfreq("h")
ax = first_week[["aktual", "hybrid"]].plot(figsize=(9, 3), color=["#444a58", "#167568"])
ax.set(xlabel="Jam target", ylabel="CO (mg/m³)")
plt.tight_layout()
plt.show()

## 5. Prediksi dari CSV sensor
Contoh berisi enam jam observasi historis untuk konfigurasi yang dihasilkan. Ganti berkas contoh dengan CSV sensor lain yang memuat kolom dan jumlah jam yang disyaratkan model.

In [ ]:
sample_path = run_dir / "contoh_sensor.csv"
print(sample_path.read_text())
inference_code = (
    "from pathlib import Path; import sys; from inference import forecast; "
    "print(forecast(Path(sys.argv[1]), Path(sys.argv[2])).to_string(index=False))"
)
subprocess.run([sys.executable, "-c", inference_code, str(sample_path), str(run_dir)], cwd=work, check=True)

## 6. Simpan artefak eksperimen
Arsip memuat bobot keempat model, parameter normalisasi, metrik, kurva, semua prediksi uji dan CSV contoh. Unduh arsip melalui panel Files jika bekerja di Colab.

In [ ]:
import shutil
destination = Path.cwd() / ("Artefak_CO_" + run_dir.name)
archive_path = Path(shutil.make_archive(str(destination), "zip", root_dir=run_dir))
print("Arsip:", archive_path.name)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive_path))